# 📊 THỬ NGHIỆM SO SÁNH HIỆU NĂNG MÔ HÌNH CHUYÊN BIỆT HÀ NỘI (HANOI-ONLY VS FULL MODELS)

**Mục tiêu:** Huấn luyện và đánh giá đối chiếu hiệu năng của mô hình dự đoán giá bất động sản khi huấn luyện chuyên biệt trên tập dữ liệu Hà Nội so với mô hình huấn luyện toàn quốc:
1. **Mô hình Full (Có City Feature):** Huấn luyện trên dữ liệu toàn quốc (gồm cả Hà Nội) có sử dụng đặc trưng `city`.
2. **Mô hình Full (Không City Feature):** Huấn luyện trên dữ liệu toàn quốc nhưng loại bỏ đặc trưng `city`.
3. **Mô hình Hanoi-Only:** Huấn luyện **chỉ** trên tập dữ liệu Hà Nội (loại bỏ cột `city` vì nó là hằng số).

**Phương pháp đánh giá:** Để đảm bảo tính khách quan và khoa học (tránh rò rỉ dữ liệu), ta lọc riêng dữ liệu Hà Nội ra trước, chia 80% Train và 20% Test. Tập Test Hà Nội (20%) này là tập kiểm thử chung cho cả 3 mô hình. Tập Train của mô hình Full sẽ kết hợp tập Train Hà Nội (80%) và tập Train của các tỉnh khác (80%) để mô hình Full không bị rò rỉ bất kỳ mẫu test nào.

## 1. Import các thư viện cần thiết và thiết lập định dạng hiển thị

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

from IPython.display import display, HTML
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
from xgboost import XGBRegressor

# Hiển thị đầy đủ nội dung bảng
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 240)
pd.set_option('display.expand_frame_repr', False)

# CSS giúp bảng trong notebook xuống dòng đầy đủ
display(HTML("""
<style>
.output_area table.dataframe {
    width: 100% !important;
    table-layout: auto !important;
}
.output_area table.dataframe th,
.output_area table.dataframe td {
    white-space: pre-wrap !important;
    word-break: break-word !important;
    text-align: left !important;
    vertical-align: middle !important;
    max-width: none !important;
}
</style>
"""))

def print_section(title: str) -> None:
    print('\n' + '=' * 90)
    print(title)
    print('=' * 90)

print_section('0. THIẾT LẬP MÔI TRƯỜNG THÀNH CÔNG')


0. THIẾT LẬP MÔI TRƯỜNG THÀNH CÔNG


## 2. Load dữ liệu từ thư mục `data/processed`

In [2]:
DATA_CC = '../data/processed/cleaned_chung_cu.csv'
DATA_ND = '../data/processed/cleaned_nha_dat.csv'

df_cc = pd.read_csv(DATA_CC)
df_nd = pd.read_csv(DATA_ND)

print_section('1. ĐỌC DỮ LIỆU ĐẦU VÀO')
print(f'Số dòng Chung cư ban đầu : {len(df_cc):,}')
print(f'Số dòng Nhà đất ban đầu  : {len(df_nd):,}')


1. ĐỌC DỮ LIỆU ĐẦU VÀO
Số dòng Chung cư ban đầu : 5,452
Số dòng Nhà đất ban đầu  : 6,379


## 3. Tiền xử lý dữ liệu và Lọc Outliers trên dữ liệu chung toàn quốc (Đồng nhất)

In [3]:
# 1. Tiền xử lý Chung cư
df_cc_clean = df_cc.copy()
if 'balcony_direction' in df_cc_clean.columns:
    df_cc_clean = df_cc_clean.drop(columns=['balcony_direction'])
df_cc_clean = df_cc_clean.dropna(subset=['price_billion','area_m2'])
df_cc_clean = df_cc_clean[(df_cc_clean['price_billion'] >= 1) & (df_cc_clean['price_billion'] <= 200)]
df_cc_clean = df_cc_clean[(df_cc_clean['area_m2'] >= 10) & (df_cc_clean['area_m2'] <= 1000)]

# 2. Tiền xử lý Nhà đất
df_nd_clean = df_nd.copy()
df_nd_clean = df_nd_clean.dropna(subset=['price_billion','area_m2'])
df_nd_clean = df_nd_clean[(df_nd_clean['price_billion'] >= 1) & (df_nd_clean['price_billion'] <= 200)]
df_nd_clean = df_nd_clean[(df_nd_clean['area_m2'] >= 10) & (df_nd_clean['area_m2'] <= 1000)]

print_section('2. KẾT QUẢ TIỀN XỬ LÝ & LỌC OUTLIERS TOÀN QUỐC')
print(f'Chung cư sạch toàn quốc : {len(df_cc_clean):,} dòng')
print(f'Nhà đất sạch toàn quốc  : {len(df_nd_clean):,} dòng')


2. KẾT QUẢ TIỀN XỬ LÝ & LỌC OUTLIERS TOÀN QUỐC
Chung cư sạch toàn quốc : 5,446 dòng
Nhà đất sạch toàn quốc  : 6,340 dòng


## 4. Lọc Hà Nội & Chia tập Train/Test (80/20) để đảm bảo không rò rỉ dữ liệu

In [4]:
def split_and_combine(df_clean):
    # Cast columns to string
    for col in ['city', 'district', 'direction', 'furniture_std', 'legal_std']:
        if col in df_clean.columns:
            df_clean[col] = df_clean[col].astype(str).str.strip()

    # 1. Lọc Hà Nội
    df_hn = df_clean[df_clean['city'] == 'Hà Nội'].copy()
    # 2. Chia Hà Nội 80/20
    df_train_hn, df_test_hn = train_test_split(df_hn, test_size=0.2, random_state=42)

    # 3. Lọc tỉnh khác và chia 80/20
    df_other = df_clean[df_clean['city'] != 'Hà Nội'].copy()
    df_train_other, df_test_other = train_test_split(df_other, test_size=0.2, random_state=42)

    # 4. Gộp lại để có tập Train/Test toàn quốc không rò rỉ tập Test Hà Nội
    df_train_full = pd.concat([df_train_hn, df_train_other], ignore_index=True)
    df_test_full = pd.concat([df_test_hn, df_test_other], ignore_index=True)

    return df_train_hn, df_test_hn, df_train_full, df_test_full

train_cc_hn, test_cc_hn, train_cc_full, test_cc_full = split_and_combine(df_cc_clean)
train_nd_hn, test_nd_hn, train_nd_full, test_nd_full = split_and_combine(df_nd_clean)

print_section('3. THÔNG TIN SỐ MẪU SAU KHI PHÂN CHIA')
print('A. Chung cư:')
print(f'  - Hà Nội: Train = {len(train_cc_hn):,} | Test = {len(test_cc_hn):,}')
print(f'  - Toàn quốc: Train = {len(train_cc_full):,} | Test = {len(test_cc_full):,}')
print('\nB. Nhà đất:')
print(f'  - Hà Nội: Train = {len(train_nd_hn):,} | Test = {len(test_nd_hn):,}')
print(f'  - Toàn quốc: Train = {len(train_nd_full):,} | Test = {len(test_nd_full):,}')


3. THÔNG TIN SỐ MẪU SAU KHI PHÂN CHIA
A. Chung cư:
  - Hà Nội: Train = 875 | Test = 219
  - Toàn quốc: Train = 4,356 | Test = 1,090

B. Nhà đất:
  - Hà Nội: Train = 2,174 | Test = 544
  - Toàn quốc: Train = 5,071 | Test = 1,269


## 5. Thiết lập Đặc trưng & Preprocessors độc lập cho 2 nhóm

In [5]:
# Đặc trưng Chung cư
num_cc = ['area_m2', 'bedrooms_num']
cat_cc_with = ['city', 'district', 'direction', 'furniture_std', 'legal_std']
cat_cc_without = ['district', 'direction', 'furniture_std', 'legal_std']

# Đặc trưng Nhà đất
num_nd = ['area_m2', 'bedrooms_num', 'floors_num', 'frontage_m', 'road_width_m']
cat_nd_with = ['city', 'district', 'direction', 'furniture_std', 'legal_std']
cat_nd_without = ['district', 'direction', 'furniture_std', 'legal_std']

# Khởi tạo các preprocessors
prep_cc_with = ColumnTransformer([('num', StandardScaler(), num_cc), ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cc_with)])
prep_cc_without = ColumnTransformer([('num', StandardScaler(), num_cc), ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cc_without)])

prep_nd_with = ColumnTransformer([('num', StandardScaler(), num_nd), ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_nd_with)])
prep_nd_without = ColumnTransformer([('num', StandardScaler(), num_nd), ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_nd_without)])

## 6. Định nghĩa Hàm Đánh giá & Huấn luyện

In [6]:
def train_and_eval_hanoi_target(model, preprocessor, X_train, y_train, X_test_hn, y_test_hn, features_list):
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    pipeline.fit(X_train[features_list], y_train)

    y_pred = pipeline.predict(X_test_hn[features_list])

    r2 = r2_score(y_test_hn, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test_hn, y_pred))
    mae = mean_absolute_error(y_test_hn, y_pred)
    mape = mean_absolute_percentage_error(y_test_hn, y_pred)

    # CV R2 Mean trên tập Train để đo mức độ ổn định
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(pipeline, X_train[features_list], y_train, cv=cv, scoring='r2', n_jobs=1)

    return {
        'R2_Test': r2,
        'RMSE_Test': rmse,
        'MAE_Test': mae,
        'MAPE_Test': mape,
        'CV_Mean': cv_scores.mean(),
        'CV_Std': cv_scores.std(),
        'pipeline': pipeline
    }

## 7. Chạy thực nghiệm huấn luyện và đánh giá

In [7]:
models_dict = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(max_depth=10, min_samples_split=5, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=15, min_samples_split=4, random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=7, subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1, verbosity=0)
}

# Kết quả Chung cư Hà Nội
res_cc_full_with = {}
res_cc_full_without = {}
res_cc_hn_only = {}

print_section('4A. CHẠY HUẤN LUYỆN MODEL CHUNG CƯ')
for name, model in models_dict.items():
    # 1. Full Model có City
    res_cc_full_with[name] = train_and_eval_hanoi_target(
        model, prep_cc_with, train_cc_full, train_cc_full['price_billion'], test_cc_hn, test_cc_hn['price_billion'], num_cc + cat_cc_with
    )
    # 2. Full Model không City
    res_cc_full_without[name] = train_and_eval_hanoi_target(
        model, prep_cc_without, train_cc_full, train_cc_full['price_billion'], test_cc_hn, test_cc_hn['price_billion'], num_cc + cat_cc_without
    )
    # 3. Hanoi Only Model
    res_cc_hn_only[name] = train_and_eval_hanoi_target(
        model, prep_cc_without, train_cc_hn, train_cc_hn['price_billion'], test_cc_hn, test_cc_hn['price_billion'], num_cc + cat_cc_without
    )
    print(f'  ✅ {name:<20} | Hanoi-Only: R²={res_cc_hn_only[name]["R2_Test"]:.4f}, MAE={res_cc_hn_only[name]["MAE_Test"]:.3f} | Full (with City): R²={res_cc_full_with[name]["R2_Test"]:.4f}, MAE={res_cc_full_with[name]["MAE_Test"]:.3f}')

# Kết quả Nhà đất Hà Nội
res_nd_full_with = {}
res_nd_full_without = {}
res_nd_hn_only = {}

print_section('4B. CHẠY HUẤN LUYỆN MODEL NHÀ ĐẤT')
for name, model in models_dict.items():
    # 1. Full Model có City
    res_nd_full_with[name] = train_and_eval_hanoi_target(
        model, prep_nd_with, train_nd_full, train_nd_full['price_billion'], test_nd_hn, test_nd_hn['price_billion'], num_nd + cat_nd_with
    )
    # 2. Full Model không City
    res_nd_full_without[name] = train_and_eval_hanoi_target(
        model, prep_nd_without, train_nd_full, train_nd_full['price_billion'], test_nd_hn, test_nd_hn['price_billion'], num_nd + cat_nd_without
    )
    # 3. Hanoi Only Model
    res_nd_hn_only[name] = train_and_eval_hanoi_target(
        model, prep_nd_without, train_nd_hn, train_nd_hn['price_billion'], test_nd_hn, test_nd_hn['price_billion'], num_nd + cat_nd_without
    )
    print(f'  ✅ {name:<20} | Hanoi-Only: R²={res_nd_hn_only[name]["R2_Test"]:.4f}, MAE={res_nd_hn_only[name]["MAE_Test"]:.3f} | Full (with City): R²={res_nd_full_with[name]["R2_Test"]:.4f}, MAE={res_nd_full_with[name]["MAE_Test"]:.3f}')


4A. CHẠY HUẤN LUYỆN MODEL CHUNG CƯ


  ✅ Linear Regression    | Hanoi-Only: R²=0.6109, MAE=2.045 | Full (with City): R²=0.6238, MAE=2.023


  ✅ Decision Tree        | Hanoi-Only: R²=0.5226, MAE=2.062 | Full (with City): R²=0.4592, MAE=2.099


  ✅ Random Forest        | Hanoi-Only: R²=0.5880, MAE=1.863 | Full (with City): R²=0.5782, MAE=1.864


  ✅ XGBoost              | Hanoi-Only: R²=0.6109, MAE=1.779 | Full (with City): R²=0.6313, MAE=1.787

4B. CHẠY HUẤN LUYỆN MODEL NHÀ ĐẤT


  ✅ Linear Regression    | Hanoi-Only: R²=0.6300, MAE=5.563 | Full (with City): R²=0.5681, MAE=6.118


  ✅ Decision Tree        | Hanoi-Only: R²=0.5213, MAE=5.616 | Full (with City): R²=0.5454, MAE=5.517


  ✅ Random Forest        | Hanoi-Only: R²=0.7057, MAE=4.752 | Full (with City): R²=0.7137, MAE=4.696


  ✅ XGBoost              | Hanoi-Only: R²=0.7391, MAE=4.416 | Full (with City): R²=0.7405, MAE=4.410


## 8. Trình bày kết quả bằng Pandas Styler (Đối chiếu hiệu năng trên tập Test Hà Nội)

In [8]:
def get_hanoi_summary_df(full_with, full_without, hn_only):
    rows = []
    for name in models_dict.keys():
        fw, fwo, ho = full_with[name], full_without[name], hn_only[name]
        rows.append({
            'Model': name,
            'Full (Có City): R2': fw['R2_Test'],
            'Full (Không City): R2': fwo['R2_Test'],
            'Hanoi-Only: R2': ho['R2_Test'],
            'R2 Diff (HN vs Full+City)': ho['R2_Test'] - fw['R2_Test'],
            'Full (Có City): MAE': fw['MAE_Test'],
            'Full (Không City): MAE': fwo['MAE_Test'],
            'Hanoi-Only: MAE': ho['MAE_Test'],
            'MAE Diff (HN vs Full+City)': ho['MAE_Test'] - fw['MAE_Test']
        })
    return pd.DataFrame(rows)

def style_hanoi_df(df):
    return df.style.format({
        'Full (Có City): R2': '{:.4f}', 'Full (Không City): R2': '{:.4f}', 'Hanoi-Only: R2': '{:.4f}', 'R2 Diff (HN vs Full+City)': '{:+.4f}',
        'Full (Có City): MAE': '{:.4f}', 'Full (Không City): MAE': '{:.4f}', 'Hanoi-Only: MAE': '{:.4f}', 'MAE Diff (HN vs Full+City)': '{:+.4f}'
    }).background_gradient(cmap='RdYlGn', subset=['R2 Diff (HN vs Full+City)'])\
      .background_gradient(cmap='RdYlGn_r', subset=['MAE Diff (HN vs Full+City)'])\
      .set_properties(**{'white-space': 'pre-wrap', 'text-align': 'left', 'vertical-align': 'middle'})

df_cc_hn_sum = get_hanoi_summary_df(res_cc_full_with, res_cc_full_without, res_cc_hn_only)
df_nd_hn_sum = get_hanoi_summary_df(res_nd_full_with, res_nd_full_without, res_nd_hn_only)

print_section('5A. BẢNG SO SÁNH HIỆU NĂNG CHUNG CƯ TẠI HÀ NỘI')
display(style_hanoi_df(df_cc_hn_sum))

print_section('5B. BẢNG SO SÁNH HIỆU NĂNG NHÀ ĐẤT TẠI HÀ NỘI')
display(style_hanoi_df(df_nd_hn_sum))


5A. BẢNG SO SÁNH HIỆU NĂNG CHUNG CƯ TẠI HÀ NỘI


,Model,Full (Có City): R2,Full (Không City): R2,Hanoi-Only: R2,R2 Diff (HN vs Full+City),Full (Có City): MAE,Full (Không City): MAE,Hanoi-Only: MAE,MAE Diff (HN vs Full+City)
0,Linear Regression,0.6238,0.6312,0.6109,-0.0129,2.0234,1.9416,2.0448,+0.0214
1,Decision Tree,0.4592,0.4230,0.5226,+0.0634,2.0989,2.1195,2.0622,-0.0367
2,Random Forest,0.5782,0.5543,0.5880,+0.0098,1.8635,1.8970,1.8633,-0.0002
3,XGBoost,0.6313,0.6126,0.6109,-0.0204,1.7874,1.7256,1.7793,-0.0081



5B. BẢNG SO SÁNH HIỆU NĂNG NHÀ ĐẤT TẠI HÀ NỘI


,Model,Full (Có City): R2,Full (Không City): R2,Hanoi-Only: R2,R2 Diff (HN vs Full+City),Full (Có City): MAE,Full (Không City): MAE,Hanoi-Only: MAE,MAE Diff (HN vs Full+City)
0,Linear Regression,0.5681,0.5677,0.6300,+0.0619,6.1180,6.1296,5.5625,-0.5555
1,Decision Tree,0.5454,0.5405,0.5213,-0.0241,5.5165,5.3143,5.6165,+0.0999
2,Random Forest,0.7137,0.7024,0.7057,-0.0080,4.6961,4.6838,4.7524,+0.0563
3,XGBoost,0.7405,0.7256,0.7391,-0.0014,4.4097,4.4015,4.4163,+0.0066


## 9. Vẽ và lưu trữ các biểu đồ so sánh chuyên biệt cho Hà Nội

In [9]:
def draw_hanoi_comparison_chart(full_with, full_without, hn_only, title, filename):
    names = list(models_dict.keys())
    x_pos = np.arange(len(names))
    width = 0.25

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    fig.suptitle(title, fontweight='bold', fontsize=16)

    # R2 Score
    axes[0].bar(x_pos - width, [full_with[n]['R2_Test'] for n in names], width, label='Full (With City)', color='#3498db', edgecolor='black')
    axes[0].bar(x_pos, [full_without[n]['R2_Test'] for n in names], width, label='Full (Without City)', color='#95a5a6', edgecolor='black')
    axes[0].bar(x_pos + width, [hn_only[n]['R2_Test'] for n in names], width, label='Hanoi-Only', color='#2ecc71', edgecolor='black')
    axes[0].set_title('R² Test on Hanoi Dataset (Higher is better ↑)', fontweight='bold')
    axes[0].set_xticks(x_pos)
    axes[0].set_xticklabels(names, rotation=15)
    axes[0].legend()
    axes[0].grid(True, axis='y', linestyle='--', alpha=0.7)

    # MAE
    axes[1].bar(x_pos - width, [full_with[n]['MAE_Test'] for n in names], width, label='Full (With City)', color='#e74c3c', edgecolor='black')
    axes[1].bar(x_pos, [full_without[n]['MAE_Test'] for n in names], width, label='Full (Without City)', color='#95a5a6', edgecolor='black')
    axes[1].bar(x_pos + width, [hn_only[n]['MAE_Test'] for n in names], width, label='Hanoi-Only', color='#f1c40f', edgecolor='black')
    axes[1].set_title('MAE Test on Hanoi Dataset (Lower is better ↓)', fontweight='bold')
    axes[1].set_xticks(x_pos)
    axes[1].set_xticklabels(names, rotation=15)
    axes[1].legend()
    axes[1].grid(True, axis='y', linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()

print_section('6. TẠO BIỂU ĐỒ ĐỐI CHIẾU CHO HÀ NỘI')
draw_hanoi_comparison_chart(res_cc_full_with, res_cc_full_without, res_cc_hn_only, 'Hanoi Apartment Comparison: Hanoi-Only vs Full Models', '../models/ml_traditional/hanoi_only/hanoi_comparison_chart_chung_cu.png')
draw_hanoi_comparison_chart(res_nd_full_with, res_nd_full_without, res_nd_hn_only, 'Hanoi Landed House Comparison: Hanoi-Only vs Full Models', '../models/ml_traditional/hanoi_only/hanoi_comparison_chart_nha_dat.png')


6. TẠO BIỂU ĐỒ ĐỐI CHIẾU CHO HÀ NỘI


## 10. Tự động xuất Báo cáo So sánh sang Markdown (`hanoi_comparison_report.md`)

In [10]:
report_md = f'# Báo cáo Thực nghiệm: So sánh Mô hình Chuyên biệt Hà Nội (Hanoi-Only) vs Mô hình Toàn quốc (Full Models)\n\n'
report_md += '## 1. Giới thiệu thử nghiệm\n'
report_md += f'Thử nghiệm này được thực hiện nhằm đánh giá độ chính xác khi dự đoán giá bất động sản **riêng tại Hà Nội** giữa mô hình được huấn luyện chuyên biệt và mô hình toàn quốc.\n'
report_md += f'- **Chung cư Hà Nội**: Huấn luyện trên {len(train_cc_hn)} mẫu và kiểm thử trên {len(test_cc_hn)} mẫu local sạch (đã lọc outliers).\n'
report_md += f'- **Nhà đất Hà Nội**: Huấn luyện trên {len(train_nd_hn)} mẫu và kiểm thử trên {len(test_nd_hn)} mẫu local sạch (đã lọc outliers).\n\n'
report_md += 'Cả 3 mô hình (Full có City, Full không City, và Hanoi-Only) được đánh giá trên **cùng tập Test Hà Nội** để đảm bảo tính khách quan khoa học, không rò rỉ dữ liệu.\n\n'

report_md += '## 2. Kết quả đối chiếu chi tiết: Mô hình CHUNG CƯ HÀ NỘI\n\n'
report_md += '| Thuật toán | Cấu hình | R² Test | RMSE (tỷ) | MAE (tỷ) | MAPE | CV R² Mean |\n'
report_md += '| :--- | :--- | :---: | :---: | :---: | :---: | :---: |\n'
for name in models_dict.keys():
    fw = res_cc_full_with[name]
    fwo = res_cc_full_without[name]
    ho = res_cc_hn_only[name]
    report_md += f"| **{name}** | Hanoi-Only | {ho['R2_Test']:.4f} | {ho['RMSE_Test']:.4f} | {ho['MAE_Test']:.4f} | {ho['MAPE_Test']:.1%} | {ho['CV_Mean']:.4f} ± {ho['CV_Std']:.4f} |\n"
    report_md += f"| | Full (Có City) | {fw['R2_Test']:.4f} | {fw['RMSE_Test']:.4f} | {fw['MAE_Test']:.4f} | {fw['MAPE_Test']:.1%} | {fw['CV_Mean']:.4f} ± {fw['CV_Std']:.4f} |\n"
    report_md += f"| | Full (Không City) | {fwo['R2_Test']:.4f} | {fwo['RMSE_Test']:.4f} | {fwo['MAE_Test']:.4f} | {fwo['MAPE_Test']:.1%} | {fwo['CV_Mean']:.4f} ± {fwo['CV_Std']:.4f} |\n"
    report_md += f"| | *Độ lệch (HN vs Full+City)* | *{ho['R2_Test'] - fw['R2_Test']:+.4f}* | *{ho['RMSE_Test'] - fw['RMSE_Test']:+.4f}* | *{ho['MAE_Test'] - fw['MAE_Test']:+.4f}* | *{ho['MAPE_Test'] - fw['MAPE_Test']:+.1%}* | |\n"
    report_md += "| | | | | | | |\n"

report_md += '\n## 3. Kết quả đối chiếu chi tiết: Mô hình NHÀ ĐẤT HÀ NỘI\n\n'
report_md += '| Thuật toán | Cấu hình | R² Test | RMSE (tỷ) | MAE (tỷ) | MAPE | CV R² Mean |\n'
report_md += '| :--- | :--- | :---: | :---: | :---: | :---: | :---: |\n'
for name in models_dict.keys():
    fw = res_nd_full_with[name]
    fwo = res_nd_full_without[name]
    ho = res_nd_hn_only[name]
    report_md += f"| **{name}** | Hanoi-Only | {ho['R2_Test']:.4f} | {ho['RMSE_Test']:.4f} | {ho['MAE_Test']:.4f} | {ho['MAPE_Test']:.1%} | {ho['CV_Mean']:.4f} ± {ho['CV_Std']:.4f} |\n"
    report_md += f"| | Full (Có City) | {fw['R2_Test']:.4f} | {fw['RMSE_Test']:.4f} | {fw['MAE_Test']:.4f} | {fw['MAPE_Test']:.1%} | {fw['CV_Mean']:.4f} ± {fw['CV_Std']:.4f} |\n"
    report_md += f"| | Full (Không City) | {fwo['R2_Test']:.4f} | {fwo['RMSE_Test']:.4f} | {fwo['MAE_Test']:.4f} | {fwo['MAPE_Test']:.1%} | {fwo['CV_Mean']:.4f} ± {fwo['CV_Std']:.4f} |\n"
    report_md += f"| | *Độ lệch (HN vs Full+City)* | *{ho['R2_Test'] - fw['R2_Test']:+.4f}* | *{ho['RMSE_Test'] - fw['RMSE_Test']:+.4f}* | *{ho['MAE_Test'] - fw['MAE_Test']:+.4f}* | *{ho['MAPE_Test'] - fw['MAPE_Test']:+.1%}* | |\n"
    report_md += "| | | | | | | |\n"

report_md += '\n## 4. Trực quan hóa biểu đồ\n'
report_md += '### A. Biểu đồ Chung cư Hà Nội\n![Biểu đồ Chung cư Hà Nội](hanoi_comparison_chart_chung_cu.png)\n\n'
report_md += '### B. Biểu đồ Nhà đất Hà Nội\n![Biểu đồ Nhà đất Hà Nội](hanoi_comparison_chart_nha_dat.png)\n\n'

report_md += '## 5. Nhận xét & Kết luận chuyên sâu (Lý giải khoa học)\n\n'
# Tính toán sự chênh lệch thực tế để tự động điền nhận xét nếu cần
with open('../models/ml_traditional/hanoi_only/hanoi_comparison_report.md', 'w', encoding='utf-8') as f:
    f.write(report_md)
print_section('7. XUẤT BÁO CÁO HANOI_COMPARISON_REPORT.MD')
print('✅ Đã xuất báo cáo hanoi_comparison_report.md thành công!')


7. XUẤT BÁO CÁO HANOI_COMPARISON_REPORT.MD
✅ Đã xuất báo cáo hanoi_comparison_report.md thành công!


In [11]:
import os
import pickle

os.makedirs('../models/ml_traditional/hanoi_only', exist_ok=True)

# Save Hanoi-only ML models
with open('../models/ml_traditional/hanoi_only/model_cc_hanoi_only.pkl', 'wb') as f:
    pickle.dump(res_cc_hn_only['XGBoost']['pipeline'], f)
with open('../models/ml_traditional/hanoi_only/model_nd_hanoi_only.pkl', 'wb') as f:
    pickle.dump(res_nd_hn_only['XGBoost']['pipeline'], f)

print('✅ Đã lưu thành công 2 file mô hình ML chuyên biệt Hà Nội (XGBoost) vào thư mục models/ml_traditional/hanoi_only/!')

✅ Đã lưu thành công 2 file mô hình ML chuyên biệt Hà Nội (XGBoost) vào thư mục models/ml_traditional/hanoi_only/!
